In [0]:
# ═══════════════════════════════════════════════════════════════════
# PIPELINE DLT — EduStream (arquitectura Medallion declarativa)
# ═══════════════════════════════════════════════════════════════════
# Bronze  → ingesta incremental de CSVs con Auto Loader (cloudFiles)
# Silver  → limpieza + expectativas de calidad (@dlt.expect_all_or_drop)
# Gold    → métricas de negocio agregadas
#
# DLT detecta automáticamente las dependencias entre tablas a partir
# de dlt.read() / dlt.read_stream() y las ejecuta en el orden correcto.
# ═══════════════════════════════════════════════════════════════════

# ── Librerías del sistema ──────────────────────────────────────────
import dlt
import requests

# ── Tipos de datos Spark ───────────────────────────────────────────
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

# ── Funciones Spark ────────────────────────────────────────────────
from pyspark.sql.functions import (
    col,
    when,
    year,
    month,
    date_format,
    countDistinct       as count_distinct,
    count               as spark_count,
    sum                 as spark_sum,
    avg                 as spark_avg,
    round               as spark_round
)

################################## CAPA BRONZE ########################################
# Auto Loader (cloudFiles) detecta archivos nuevos automáticamente y los procesa
# de forma incremental (streaming). schemaLocation guarda el esquema inferido para
# mantener consistencia entre ejecuciones. pathGlobFilter lee solo el CSV indicado.

@dlt.table(name='bronze_enrollments_dlt')
def bronze_enrollments_dlt():
    return (spark.readStream
        .format('cloudFiles')
        .option('cloudFiles.format', 'csv')
        .option('cloudFiles.schemaLocation', '/Volumes/workspace/edustream/landing/_schema/enrollments')
        .option('cloudFiles.inferColumnTypes', 'true')
        .option('header', 'true')
        .option('pathGlobFilter', 'enrollments.csv')
        .load('/Volumes/workspace/edustream/landing/'))    
    
@dlt.table(name='bronze_instructors_dlt')
def bronze_instructors_dlt():
    return (spark.readStream
        .format('cloudFiles')
        .option('cloudFiles.format', 'csv')
        .option('cloudFiles.schemaLocation', '/Volumes/workspace/edustream/landing/_schema/instructors')
        .option('cloudFiles.inferColumnTypes', 'true')
        .option('header', 'true')
        .option('pathGlobFilter', 'instructors.csv')
        .load('/Volumes/workspace/edustream/landing/'))  

@dlt.table(name='bronze_courses_dlt')
def bronze_courses_dlt():
    return (spark.readStream
        .format('cloudFiles')
        .option('cloudFiles.format', 'csv')
        .option('cloudFiles.schemaLocation', '/Volumes/workspace/edustream/landing/_schema/courses')
        .option('cloudFiles.inferColumnTypes', 'true')
        .option('header', 'true')
        .option('pathGlobFilter', 'courses.csv')
        .load('/Volumes/workspace/edustream/landing/'))
        
@dlt.table(name='bronze_progress_dlt')
def bronze_progress_dlt():
    return (spark.readStream
        .format('cloudFiles')
        .option('cloudFiles.format', 'csv')
        .option('cloudFiles.schemaLocation', '/Volumes/workspace/edustream/landing/_schema/progress')
        .option('cloudFiles.inferColumnTypes', 'true')
        .option('header', 'true')
        .option('pathGlobFilter', 'progress.csv')
        .load('/Volumes/workspace/edustream/landing/'))
    

################################## CAPA SILVER ########################################

#--------------------------------------------------------------------------------------
# Función para pedir a la API el tipo de cambio

def get_exchange_rate(currency):
    if currency == 'USD':
        return 1.0
    try:
        url = f"https://open.er-api.com/v6/latest/{currency}"
        response = requests.get(url, timeout=5)
        return float(response.json()['rates']['USD'])
    except Exception as e:
        #print(f"⚠️ Error con {currency}: {e}")
        return 1.0
#--------------------------------------------------------------------------------------

@dlt.table(name='silver_enrollments_dlt')
@dlt.expect_all_or_drop({
    "enrollment_id válido" : "enrollment_id IS NOT NULL",
    "user_id válido"       : "user_id IS NOT NULL",
    "course_id válido"     : "course_id IS NOT NULL",
    "enrolled_at válido"   : "enrolled_at IS NOT NULL",
    "pago válido"          : "payment_amount >= 0"
})
def silver_enrollments_dlt():
    # read_stream: procesa solo los registros nuevos de Bronze (incremental).
    # _rescued_data: columna que Auto Loader agrega con datos que no encajaron
    # en el esquema; la eliminamos para evitar conflictos en los joins posteriores.
    return (dlt.read_stream('bronze_enrollments_dlt')
        .drop('_rescued_data')    
        )


@dlt.table(name='silver_instructors_dlt')
@dlt.expect_all_or_drop({
    "instructor_id válido" : "instructor_id IS NOT NULL",
    "name válido"          : "name IS NOT NULL",
    "joined_at válido"     : "joined_at IS NOT NULL"
})
def silver_instructors_dlt():
    return (dlt.read_stream('bronze_instructors_dlt')
        .fillna({'country': 'unknown'})
        .drop('_rescued_data')
    )


@dlt.table(name='silver_courses_dlt')
@dlt.expect_all_or_drop({
    "course_id válido" : "course_id IS NOT NULL",
    "title válido"     : "title IS NOT NULL",
    "instructor_id válido" : "instructor_id IS NOT NULL",
    "price_usd válido" : "price_usd >= 0",
    "language válido"    : "language IS NOT NULL"
})
def silver_courses_dlt():
    return (dlt.read_stream('bronze_courses_dlt')
        .fillna({'category': 'other'})
        .drop('_rescued_data')
    )


@dlt.table(name='silver_progress_dlt')
@dlt.expect_all_or_drop({
    "user_id válido"     : "user_id IS NOT NULL",
    "course_id válido"   : "course_id IS NOT NULL",
    "total_lessons válido": "total_lessons IS NOT NULL AND total_lessons > 0",
    "last_activity_at válido": "last_activity_at IS NOT NULL"
})
def silver_progress_dlt():
    return (dlt.read_stream('bronze_progress_dlt')
        .fillna({'lessons_completed': 0})
        .drop('_rescued_data')
    )


@dlt.table(name='silver_progress_clean_dlt')
def silver_progress_clean_dlt():
    # read (batch, no stream): necesario porque hacemos un JOIN entre dos tablas.
    # Los joins entre streams son complejos; en batch se resuelven directamente.
    df_progress =dlt.read('silver_progress_dlt')
    df_courses = dlt.read('silver_courses_dlt')
    return (df_progress
        .withColumn('completion_rate',
            spark_round(col('lessons_completed') / col('total_lessons'), 4))
        .join(df_courses, on=['course_id'], how='left')
    )


@dlt.table(name='silver_currency_dlt')
def silver_currency_dlt():
    # Obtiene las monedas únicas presentes en enrollments y consulta su tasa a USD.
    # NOTA: collect() trae los datos al driver — DLT muestra un warning porque no es
    # lo ideal en pipelines declarativos, pero aquí es seguro porque son pocas monedas
    # (5-6). Se hace 1 llamada a la API por moneda, no por fila.
    currencies = [row.currency for row in 
        dlt.read('silver_enrollments_dlt')
            .select('currency')
            .distinct()
            .collect()
        ]
    rates_data = [(c,get_exchange_rate(c)) for c in currencies]
    schema = StructType([
        StructField('currency', StringType()), 
        StructField('exchange_rate', DoubleType())
    ])
    return spark.createDataFrame(rates_data,schema)


@dlt.table(name='silver_enrollments_clean_dlt')
def silver_enrollments_clean_dlt():
    # Enriquecimiento progresivo:
    #   + currency    → para convertir payment_amount a USD
    #   + courses     → trae category, title, instructor_id
    #   + instructors → trae name, country (usa el instructor_id que vino de courses)
    df_enrollments = dlt.read('silver_enrollments_dlt')
    df_currency = dlt.read('silver_currency_dlt')
    df_courses = dlt.read('silver_courses_dlt')
    df_instructors = dlt.read('silver_instructors_dlt')

    return (df_enrollments
        .join(df_currency, on=['currency'], how='left')
        .join(df_courses, on=['course_id'], how='left')
        .join(df_instructors, on=['instructor_id'], how='left')
        .withColumn('payment_USD', spark_round(col('payment_amount') * col('exchange_rate'), 6))
        .drop('payment_amount', 'exchange_rate')
    )


################################## CAPA GOLD ########################################

# 1. Completion Rate por curso 
# Métrica: % de usuarios que completaron el 100% de las lecciones

@dlt.table(name='gold_completion_rate_by_course_dlt')
def gold_completion_rate_dlt():
    df_progress = dlt.read('silver_progress_clean_dlt')
    return (df_progress
        .groupBy('course_id','title','category')
        .agg(
            spark_count('user_id').alias('total_users'),
            spark_sum(when(col('completion_rate') == 1.0, 1)
                .otherwise(0)).alias('total_users_completed')
        )
        .withColumn(
            'pct_users_completed',
            spark_round(col('total_users_completed') / col('total_users') * 100, 2)
        )
        .orderBy(col('pct_users_completed').desc())
    )



#2. revenue total generado por instructor
# Métrica: suma de ingresos en USD de todos los cursos de cada instructor

@dlt.table(name='gold_revenue_by_instructor_dlt')
def gold_revenue_by_instructor_dlt():
    df_enrollments = dlt.read('silver_enrollments_clean_dlt')
    return (df_enrollments
        .groupBy('instructor_id','name','country')
        .agg(
            spark_sum('payment_USD').alias('total_revenue_USD'),
            spark_count('enrollment_id').alias('total_enrollments'),
            count_distinct('course_id').alias('total_courses')
        )
        .withColumn('total_revenue_USD', spark_round(col('total_revenue_USD'), 6))
        .orderBy(col('total_revenue_USD').desc())
    )


# 3. Gold: cursos con alta inscripción y bajo completion rate ─────
# Se requiere mínimo de inscripciones para que la métrica sea estadísticamente relevante

@dlt.table(name='gold_top_abandoned_courses_dlt')
def gold_top_abandoned_courses_dlt():

    df_progress     = dlt.read('silver_progress_clean_dlt')
    df_enrollments  = dlt.read('silver_enrollments_clean_dlt')

    # a. Métricas de progreso por curso
    df_progress_by_course = (df_progress
        .groupBy('course_id', 'title', 'category')
        .agg(
            spark_avg(col('completion_rate')).alias('avg_completion_rate'),
            spark_count('user_id').alias('users_with_progress')
        )
    )

    # b. Total de inscripciones por curso
    df_enrollments_by_course = (df_enrollments
        .groupBy('course_id')
        .agg(spark_count('enrollment_id').alias('total_enrolled'))
    )

    # c. Join + cálculo de tasa de abandono
    return (df_progress_by_course
        .join(df_enrollments_by_course, on=['course_id'], how='left')
        .withColumn('avg_completion_rate', spark_round(col('avg_completion_rate'), 4))
        .withColumn('abandonment_rate',    spark_round(1 - col('avg_completion_rate'), 4))
        .filter(col('total_enrolled') >= 10)
        .orderBy(
            col('abandonment_rate').desc(),
            col('total_enrolled').desc()
        )
    )



#4. Gold: top categorías por ingresos agrupados por mes
# Métrica: suma de payment_USD por categoría y mes de inscripción

@dlt.table(name='gold_top_revenue_by_category_dlt')
def gold_top_revenue_by_category_dlt():

    df_enrollments = dlt.read('silver_enrollments_clean_dlt')

    return (df_enrollments
        .filter(col('enrolled_at').isNotNull())
        .withColumn('year',        year(col('enrolled_at')))
        .withColumn('month',       month(col('enrolled_at')))
        .withColumn('month_label', date_format(col('enrolled_at'), 'yyyy-MM'))
        .groupBy('year', 'month', 'month_label', 'category')
        .agg(
            spark_sum('payment_USD').alias('total_revenue_USD'),
            spark_count('enrollment_id').alias('total_enrollments'),
            spark_avg('payment_USD').alias('avg_revenue_per_enrollment')
        )
        .withColumn('total_revenue_USD',          spark_round(col('total_revenue_USD'), 2))
        .withColumn('avg_revenue_per_enrollment', spark_round(col('avg_revenue_per_enrollment'), 2))
        .orderBy(
            col('year').desc(),
            col('month').desc(),
            col('total_revenue_USD').desc()
        )
    )